# Bloque 1 — Fundamentos  
## Tema 3 — Introducción a los dispositivos para electrónica de potencia

En los temas anteriores se han estudiado herramientas matemáticas y formas de onda. Antes de comenzar el análisis de circuitos de conmutación, conviene introducir dos dispositivos fundamentales:

- el **diodo**;
- el **transistor MOSFET de canal N**.

El objetivo de este tema **no es todavía analizar convertidores ni circuitos completos**, sino comprender de forma sencilla cómo se comportan estos dispositivos y cómo pueden representarse mediante modelos básicos.

### Objetivos

Al finalizar este tema se espera que el estudiante pueda:

1. reconocer el comportamiento unidireccional de un diodo;
2. distinguir entre un modelo ideal, un modelo con caída constante y un modelo exponencial;
3. interpretar una curva corriente–tensión \(I\!-\!V\);
4. comprender el papel de la tensión \(V_{GS}\) en un NMOS;
5. interpretar de forma introductoria las regiones de corte y conducción;
6. utilizar Python y ngspice para visualizar características estáticas de dispositivos.

# 1. El diodo

El diodo es un dispositivo semiconductor de dos terminales que permite el paso de corriente principalmente en una dirección.

Sus terminales se denominan **ánodo** y **cátodo**. Definimos:

$$
v_D=v_A-v_K
$$

y tomamos \(i_D\) positiva desde el ánodo hacia el cátodo.

De forma cualitativa:

- si \(v_D\) es suficientemente positivo, el diodo conduce;
- si \(v_D\) es negativo, la corriente es muy pequeña y el diodo se considera en corte.

## Modelo ideal

$$
i_D=0
\qquad \text{si el diodo está OFF}
$$

y

$$
v_D=0
\qquad \text{si el diodo está ON}.
$$

## Modelo de caída constante

Una aproximación práctica sencilla consiste en considerar que el diodo comienza a conducir cuando

$$
v_D \gtrsim V_\gamma .
$$

Para un diodo de silicio puede utilizarse inicialmente:

$$
V_\gamma \approx 0.7\ \text{V}.
$$

## Modelo exponencial

Una representación más física viene dada por la ecuación de Shockley:

$$
\boxed{
i_D=I_S
\left(
e^{\frac{v_D}{nV_T}}-1
\right)
}
$$

donde:

- \(I_S\) es la corriente de saturación;
- \(n\) es el factor de idealidad;
- \(V_T\) es la tensión térmica.

A temperatura ambiente:

$$
V_T\approx25.85\ \text{mV}.
$$

Este modelo se utilizará solamente para visualizar la forma de la característica \(I\!-\!V\).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parámetros del modelo de Shockley
Is = 1e-12        # A
n  = 1.8
VT = 25.85e-3     # V

# Tensión aplicada al diodo
vD = np.linspace(-0.5, 0.9, 1500)

# Corriente del diodo
iD = Is * (np.exp(vD/(n*VT)) - 1)

# Se limita únicamente la representación para que
# la zona exponencial no domine completamente la gráfica.
iD_plot = np.clip(iD, -1e-3, 0.1)

plt.figure(figsize=(9,5))
plt.plot(vD, iD_plot)
plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)

plt.xlabel("Tensión del diodo, $v_D$ [V]")
plt.ylabel("Corriente del diodo, $i_D$ [A]")
plt.title("Característica I-V aproximada de un diodo")
plt.grid(True)
plt.show()

## Interpretación

La característica presenta dos comportamientos principales.

### Polarización inversa

Para

$$
v_D<0
$$

la corriente permanece muy próxima a cero dentro del modelo utilizado.

### Polarización directa

Cuando \(v_D\) aumenta en sentido positivo, la corriente crece rápidamente.

Por ello, en análisis elementales suele reemplazarse la característica exponencial por modelos más sencillos, por ejemplo:

$$
i_D\approx0
\qquad
v_D<V_\gamma.
$$

La elección del modelo depende del nivel de precisión requerido.

# 2. Introducción al MOSFET de canal N

Un MOSFET de canal N tiene tres terminales principales:

- **Gate** o compuerta \(G\);
- **Drain** o drenador \(D\);
- **Source** o fuente \(S\).

En una primera aproximación, su conducción entre drain y source está controlada por:

$$
V_{GS}=V_G-V_S.
$$

Existe una tensión característica denominada **tensión umbral** \(V_{TH}\).

De forma simplificada:

$$
V_{GS}<V_{TH}
\quad\Rightarrow\quad
\text{MOSFET en corte}
$$

mientras que:

$$
V_{GS}>V_{TH}
\quad\Rightarrow\quad
\text{el MOSFET puede conducir}.
$$

> En este tema no se estudia todavía un MOSFET de potencia ni sus pérdidas de conmutación. El objetivo es únicamente comprender la idea básica de control mediante \(V_{GS}\).

Como modelo pedagógico, en una región idealizada de saturación puede escribirse:

$$
\boxed{
I_D=
\frac{k_n}{2}
(V_{GS}-V_{TH})^2
}
\qquad
V_{GS}>V_{TH}.
$$

Si

$$
V_{GS}\le V_{TH},
$$

se aproxima:

$$
I_D=0.
$$

Este modelo cuadrático sirve para introducir el concepto, aunque no pretende representar con precisión un transistor moderno.

In [ ]:
# ============================================================
# Modelo pedagógico de transferencia de un NMOS
# ============================================================

Vth = 2.0      # V
kn  = 0.5      # A/V^2

VGS = np.linspace(0, 5, 500)

ID = np.where(
    VGS > Vth,
    0.5 * kn * (VGS - Vth)**2,
    0.0
)

plt.figure(figsize=(9,5))
plt.plot(VGS, ID)
plt.axvline(Vth, linestyle="--", label=r"$V_{TH}$")

plt.xlabel(r"$V_{GS}$ [V]")
plt.ylabel(r"$I_D$ [A]")
plt.title("Característica de transferencia simplificada de un NMOS")
plt.grid(True)
plt.legend()
plt.show()

## Interpretación

La gráfica anterior no representa todavía un circuito de conmutación. Únicamente muestra cómo cambia la corriente de drenador del modelo al modificar \(V_{GS}\).

### Corte

$$
V_{GS}\le V_{TH}
$$

y, en el modelo idealizado,

$$
I_D\approx0.
$$

### Conducción

Cuando

$$
V_{GS}>V_{TH},
$$

se forma un canal y el dispositivo puede conducir corriente.

En electrónica de potencia, más adelante se utilizará el MOSFET principalmente como **interruptor controlado**. Ese análisis se realizará después de introducir formalmente los circuitos de conmutación.

# 3. Caracterización con ngspice

ngspice también puede utilizarse para obtener curvas estáticas de los dispositivos.

En este tema se utilizarán únicamente **bancos de prueba de caracterización DC**.

No se estudian todavía:

- transitorios de conmutación;
- inductores o condensadores dentro de convertidores;
- pérdidas de switching;
- convertidores Buck, Boost o Buck-Boost.

La finalidad es comparar modelos simples de Python con modelos de dispositivo de SPICE.

## 3.1 Barrido DC de un diodo

El archivo `01_diode_iv.sp` aplica diferentes valores de tensión al diodo y calcula su corriente.

El comando:

```text
.dc Vtest -0.5 0.9 1m
```

hace variar la fuente \(V_{test}\) desde \(-0.5\) V hasta \(0.9\) V.

Los resultados se guardan en un archivo de texto para analizarlos posteriormente con Python.

In [ ]:
# ============================================================
# Lectura del resultado de ngspice: diodo
# ============================================================
#
# Ejecutar previamente:
#
# ngspice 01_diode_iv.sp
#
# Luego inspeccionar las columnas:

# data = np.loadtxt("diode_iv.dat")
# print(data.shape)
# print(data[:5])

# Una vez verificadas las columnas puede realizarse la gráfica.

## 3.2 Barrido de \(V_{GS}\) de un NMOS

En el segundo banco de prueba se mantiene una tensión fija entre drain y source y se realiza un barrido:

$$
V_{GS}:0\ \text{V}\rightarrow5\ \text{V}.
$$

De esta forma se obtiene una característica de transferencia:

$$
I_D=f(V_{GS}).
$$

El objetivo no es representar todavía un MOSFET de potencia real, sino visualizar que la tensión de compuerta controla la conducción.

# 4. Comparación de los dispositivos

| Dispositivo | Variable principal | Comportamiento introductorio |
|---|---|---|
| Diodo | Polaridad de \(v_D\) | Conducción principalmente en una dirección |
| NMOS | \(V_{GS}\) | La compuerta controla la conducción drain-source |

El diodo puede considerarse un dispositivo de conducción **no controlada externamente**, mientras que el MOSFET permite controlar su estado mediante una señal aplicada a la compuerta.

Esta diferencia será fundamental cuando se estudien posteriormente circuitos de electrónica de potencia.

# 5. Conclusiones

En este tema se han introducido dos dispositivos fundamentales sin estudiar todavía convertidores ni circuitos dinámicos.

## Diodo

- presenta una característica \(I\!-\!V\) no lineal;
- permite corriente principalmente en una dirección;
- puede representarse mediante modelos de diferente complejidad.

## MOSFET

- su conducción está controlada por \(V_{GS}\);
- por debajo de una tensión umbral se aproxima como dispositivo en corte;
- por encima de la tensión umbral puede conducir corriente;
- posteriormente podrá utilizarse como interruptor controlado.

Las curvas obtenidas con Python y los barridos DC de ngspice permiten caracterizar los dispositivos de manera aislada.

En el siguiente tema se comenzará a estudiar cómo estos elementos interactúan con resistencias, inductores, condensadores e interruptores dentro de circuitos.